# 04 · Agent evaluation

After this notebook you can evaluate a tool-calling agent the way interviewers expect. That means task success checked against the result, tool-selection and argument accuracy, trajectory match (including an edit-distance score), steps, tokens, cost and latency per task, and recovery from tool errors. You will also be able to write golden-trajectory regression tests and explain pass@k versus pass^k.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min (about 35 short LLM calls with a real provider; roughly one cent on gpt-4.1-mini) · **Needs:** [Function calling](../08_llm_apps/04_function_calling.ipynb), [Agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb)

## Why this matters in interviews

- "How do you evaluate an agent, as opposed to a model?" (`ag21`, `ev06`) is standard for agentic-AI roles. The answer is a *trajectory*, not one output.
- "How do you test an agent in CI?" (`ag28`) needs a concrete plan: deterministic tool mocks, a task suite with checkable end states, a pass-rate gate.
- Cost per **successful** task, recovery from tool errors and step caps are the follow-ups that separate demo builders from people who ran agents in production (`ag19`, `tl22`).
- Knowing what τ-bench, SWE-bench, GAIA and WebArena measure, and what their numbers do *not* tell you, comes up when a team is choosing a model.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag21` | How do you evaluate an agent, as opposed to a model? |
| `ev06` | What is the difference between evaluating a model and evaluating an agent? |
| `ag28` | How do you test an agent in CI? |
| `ag19` | How do you handle a tool that fails? |
| `ag06` | Two agents, same model, same tools, similar prompts. One resolves 71% of tickets, the other 38%. What explains it? |
| `ev22` | What is regression testing for an LLM system, and how is it different? |
| `ev23` | How do you evaluate latency and cost as part of quality? |
| `ev24` | What is an eval harness, practically? |
| `tl22` | Give me the four guards that stop an agent costing $400 in one run. |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import ast
import copy
import hashlib
import json
import operator
import re
import time
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from llm_setup import OFFLINE

if OFFLINE:
    OFFLINE.latency = 0.02        # pretend each offline LLM call takes 20 ms, so the latency columns mean something
pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 50)
rng = np.random.default_rng(0)

## What to measure

| Dimension | Metric | How it is checked |
|---|---|---|
| **Task success** | did the task get done? | against the **end state** or the facts in the answer, not the agent's claim |
| **Tool selection** | right tools called, no unnecessary ones | expected vs actual tool names |
| **Argument correctness** | right arguments | per-tool comparison; semantic where possible (evaluate `12*12`, don't string-compare it) |
| **Trajectory** | right sequence of calls | exact match, unordered match, or edit distance between sequences |
| **Efficiency** | steps, tokens, cost, latency per task | from API `usage` and timers; report cost per **successful** task |
| **Robustness** | recovery from tool errors | inject failures; did it still succeed? |
| **Reliability** | same task, repeated runs | pass^k: *all* k attempts succeed |
| **Safety / termination** | no forbidden tools, stops before the step cap | policy checks on the trajectory |

**In plain English:** a model eval grades one answer. An agent eval grades a *journey*: where it went, how long it took, what it cost, and whether it arrived (`ev06`). You also need a **deterministic environment**, meaning mocked tools with fixed data. Otherwise you are measuring the weather in your dependencies instead of your agent (`ag28`).

## 1. The environment: three mocked tools

Fixed data, a few milliseconds of simulated latency, and a **fault injector**. We can make a tool fail with a transient timeout on its next call, the kind of failure a retry fixes.

In [ ]:
WEATHER = {"paris": (18, "sunny"), "london": (12, "light rain"), "tokyo": (22, "cloudy"), "berlin": (15, "windy"),
           "new york": (20, "clear")}
ORDERS = {"ORD-1042": {"status": "shipped", "eta": "2026-10-02"}, "ORD-2077": {"status": "out for delivery", "eta": "today"}}

class TransientToolError(Exception):
    """A timeout-style failure that a retry can fix."""

FAULTS = {}                  # tool name -> number of upcoming calls that should time out
FLAKY = {"p": 0.0, "rng": np.random.default_rng(0)}   # random transient failures (used in section 7)

def maybe_fail(tool):
    if FAULTS.get(tool, 0) > 0:
        FAULTS[tool] -= 1
        raise TransientToolError(f"{tool}: upstream timed out")
    if FLAKY["p"] and FLAKY["rng"].random() < FLAKY["p"]:
        raise TransientToolError(f"{tool}: upstream timed out")

def get_weather(city: str) -> dict:
    maybe_fail("get_weather"); time.sleep(0.01)
    if city.lower() not in WEATHER:
        return {"error": f"unknown city '{city}'. Known cities: {', '.join(c.title() for c in WEATHER)}"}
    t, cond = WEATHER[city.lower()]
    return {"city": city.title(), "temp_c": t, "conditions": cond}

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg}
def safe_eval(expr: str) -> float:
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.operand))
        raise ValueError(f"not an arithmetic expression: {expr!r}")
    return ev(ast.parse(expr, mode="eval"))

def calculator(expression: str) -> dict:
    maybe_fail("calculator"); time.sleep(0.005)
    try:
        v = safe_eval(expression)
        return {"expression": expression, "result": int(v) if float(v).is_integer() else round(v, 6)}
    except Exception as e:
        return {"error": f"{e}. Send plain arithmetic such as '0.15 * 80'."}

def lookup_order(order_id: str) -> dict:
    maybe_fail("lookup_order"); time.sleep(0.015)
    order = ORDERS.get(order_id.strip().upper())
    return {"order_id": order_id, **order} if order else {"error": f"no order '{order_id}'. Order ids look like ORD-1042."}

REGISTRY = {"get_weather": get_weather, "calculator": calculator, "lookup_order": lookup_order}

def tool_schema(name, description, param, pdesc):
    return {"type": "function", "function": {"name": name, "description": description, "parameters": {
        "type": "object", "properties": {param: {"type": "string", "description": pdesc}},
        "required": [param], "additionalProperties": False}}}

TOOLS = [tool_schema("get_weather", "Get the current weather forecast for a city.", "city", "City name, e.g. Paris"),
         tool_schema("calculator", "Evaluate an arithmetic expression such as 17 * 23.", "expression", "Plain arithmetic"),
         tool_schema("lookup_order", "Look up the shipping status of a customer order by its order id, e.g. ORD-1042.",
                     "order_id", "Order id such as ORD-1042")]
print(get_weather("Paris"), calculator("17 * 23"), lookup_order("ORD-1042"), sep="\n")

## 2. The agent, with record/replay of LLM calls

A plain tool-calling loop over Chat Completions, with a **step cap**. Every LLM call goes through `llm()`, which does three things:
- **Caches** responses keyed on the exact request. Re-running an unchanged task during an eval costs nothing, and a *changed* prompt or tool list is automatically a cache miss. It is the same idea as recorded fixtures in CI (`ag28`).
- **Replays** the recorded latency, so cached runs still report realistic timings.
- **Counts** real calls against a budget, a simple cost ceiling (`tl22`).

The loop records everything the metrics need: tool calls with arguments, steps, tokens, latency split into LLM and tool time, errors and retries.

In [ ]:
CACHE, CALLS = {}, {"n": 0, "budget": 60}

def llm(**kwargs):
    key = hashlib.sha256(json.dumps(kwargs, sort_keys=True, default=str).encode()).hexdigest()
    if key not in CACHE:
        if CALLS["n"] >= CALLS["budget"]:
            raise RuntimeError("LLM call budget exhausted - a cost ceiling, not a bug")
        t0 = time.perf_counter()
        CACHE[key] = (client.chat.completions.create(**kwargs), time.perf_counter() - t0)
        CALLS["n"] += 1
    return CACHE[key]

SYSTEM = "You are a helpful assistant. Use the tools when they help, then answer briefly using the tool results."

def call_tool(name, args, retry_transient):
    """Execute one tool call; optionally retry once on a transient failure."""
    for attempt in (1, 2):
        try:
            return REGISTRY[name](**args), attempt - 1
        except TransientToolError as e:
            if not retry_transient or attempt == 2:
                return {"error": f"{e}. The tool may work if called again."}, attempt - 1
        except Exception as e:                                  # wrong/missing arguments: not retryable
            return {"error": f"bad arguments for {name}: {e}"}, attempt - 1

def run_agent(task, tools=TOOLS, retry_transient=False, max_steps=5):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": task}]
    tr = {"calls": [], "steps": 0, "prompt_tokens": 0, "completion_tokens": 0, "llm_s": 0.0, "tool_s": 0.0,
          "tool_errors": 0, "retries": 0, "answer": None, "hit_step_cap": False}
    for _ in range(max_steps):
        resp, latency = llm(model=MODEL, messages=messages, tools=tools, temperature=0)
        tr["steps"] += 1; tr["llm_s"] += latency
        tr["prompt_tokens"] += resp.usage.prompt_tokens; tr["completion_tokens"] += resp.usage.completion_tokens
        msg = resp.choices[0].message
        if not msg.tool_calls:
            tr["answer"] = msg.content or ""
            return tr
        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
            {"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
            for c in msg.tool_calls]})
        for c in msg.tool_calls:
            args = json.loads(c.function.arguments or "{}")
            t0 = time.perf_counter()
            result, retried = call_tool(c.function.name, args, retry_transient) if c.function.name in REGISTRY else \
                ({"error": f"unknown tool {c.function.name}"}, 0)
            tr["tool_s"] += time.perf_counter() - t0
            tr["retries"] += retried; tr["tool_errors"] += "error" in result
            tr["calls"].append((c.function.name, args))
            messages.append({"role": "tool", "tool_call_id": c.id, "content": json.dumps(result)})
    tr["hit_step_cap"] = True                                   # a named, measurable failure mode
    return tr

demo = run_agent("Is it sunny in Tokyo today?")        # not a suite task, so its warm-up time stays out of the results
print(json.dumps({k: v for k, v in demo.items()}, default=str, indent=1)[:600])

## 3. The eval suite: 9 tasks with expected tools, arguments and facts

Each case says which calls a good agent makes, which facts must appear in the final answer (our "end state"), and whether a fault is injected. The cases cover the failure classes on purpose: parallel calls, two different tools, a lowercase entity, a percentage (no arithmetic symbol to hint at the calculator), a no-tool small-talk request, and a transient outage.

In [ ]:
SUITE = [  # id, task, expected calls (name, args), facts the answer must contain, injected faults
    {"id": "T1", "task": "What is the weather in Paris right now?", "expect": [("get_weather", {"city": "Paris"})], "facts": ["18"]},
    {"id": "T2", "task": "What is 17 * 23?", "expect": [("calculator", {"expression": "17*23"})], "facts": ["391"]},
    {"id": "T3", "task": "Where is my order ORD-1042?", "expect": [("lookup_order", {"order_id": "ORD-1042"})], "facts": ["shipped"]},
    {"id": "T4", "task": "Compare the weather in London and Tokyo.",
     "expect": [("get_weather", {"city": "London"}), ("get_weather", {"city": "Tokyo"})], "facts": ["12", "22"]},
    {"id": "T5", "task": "What's the weather in Berlin, and what is 12 * 12?",
     "expect": [("get_weather", {"city": "Berlin"}), ("calculator", {"expression": "12*12"})], "facts": ["15", "144"]},
    {"id": "T6", "task": "Has order ORD-2077 shipped yet?", "expect": [("lookup_order", {"order_id": "ORD-2077"})],
     "facts": ["out for delivery"], "faults": {"lookup_order": 1}},
    {"id": "T7", "task": "what's the weather like in new york?", "expect": [("get_weather", {"city": "New York"})], "facts": ["20"]},
    {"id": "T8", "task": "Hi! Can you tell me a joke?", "expect": [], "facts": []},
    {"id": "T9", "task": "What is 15% of 80?", "expect": [("calculator", {"expression": "0.15*80"})], "facts": ["12"]},
]
print(f"{len(SUITE)} tasks; {sum(len(c['expect']) for c in SUITE)} expected tool calls")

## 4. Scoring a trajectory

- **Argument match** is per tool. City and order ids are normalised (case, whitespace). Calculator arguments are compared by **value**: `0.15*80`, `80 * 0.15` and `15/100*80` are all correct.
- **Trajectory edit distance:** Levenshtein distance between the expected and actual sequences of tool names. Similarity = 1 − distance / max(length), and two empty trajectories get 1.
- **Exact** trajectory match demands the same calls, with matching arguments, in the same order. **Unordered** match treats parallel calls as a multiset.
- **Success** means every expected fact appears in the final answer as a whole token. For the no-tool case, it means no tool was called and there was a non-empty reply.

Hand check for the edit distance: expected [search, lookup, refund] vs actual [search, search, lookup] needs 2 edits (insert one search, drop refund).

In [ ]:
def levenshtein(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

def args_match(name, got, want):
    try:
        if name == "calculator":
            return abs(safe_eval(got["expression"]) - safe_eval(want["expression"])) < 1e-9
        return {k: str(v).strip().lower() for k, v in got.items()} == {k: str(v).strip().lower() for k, v in want.items()}
    except Exception:
        return False

def has_fact(answer, fact):
    """Whole-token match: '50' must not match inside '250' (a real false positive; see Try it yourself 1)."""
    return re.search(rf"(?<![\w.]){re.escape(fact.lower())}(?![\w])", answer) is not None

def score(case, tr):
    exp_names, got_names = [n for n, _ in case["expect"]], [n for n, _ in tr["calls"]]
    matched, pool = 0, list(tr["calls"])
    for name, want in case["expect"]:                     # greedy one-to-one matching of expected calls
        hit = next((c for c in pool if c[0] == name and args_match(name, c[1], want)), None)
        if hit:
            matched += 1; pool.remove(hit)
    answer = (tr["answer"] or "").lower()
    success = (all(has_fact(answer, f) for f in case["facts"]) and not tr["hit_step_cap"]) if case["expect"] else \
              (not tr["calls"] and bool(answer))
    return {"success": success,
            "tool selection": sorted(exp_names) == sorted(got_names),
            "arg accuracy": matched / len(case["expect"]) if case["expect"] else float(not tr["calls"]),
            "exact trajectory": len(tr["calls"]) == len(case["expect"]) and all(
                g[0] == w[0] and args_match(w[0], g[1], w[1]) for g, w in zip(tr["calls"], case["expect"])),
            "unordered trajectory": sorted(exp_names) == sorted(got_names) and matched == len(case["expect"]),
            "trajectory sim": 1.0 if not exp_names and not got_names else
                              1 - levenshtein(exp_names, got_names) / max(len(exp_names), len(got_names))}

assert levenshtein(["search", "lookup", "refund"], ["search", "search", "lookup"]) == 2
assert has_fact("the result is 50.", "50") and not has_fact("can't answer 'what is 250 / 5'", "50")
assert args_match("calculator", {"expression": "80 * 0.15"}, {"expression": "0.15*80"})
assert args_match("get_weather", {"city": " paris "}, {"city": "Paris"}) and not args_match("get_weather", {"city": "London"}, {"city": "Paris"})
print("scoring helpers pass their hand checks")

## 5. Run the suite: agent v1 (no retries) vs agent v2 (retry transient errors once)

Same model, same tools, same prompt. The only difference is the harness: v2 retries a timed-out tool call once. Before running, **predict** which tasks each version fails.

In [ ]:
PRICE = {"input": 0.40, "output": 1.60}          # USD per 1M tokens, gpt-4.1-mini list price (check current pricing)

def run_suite(retry_transient, tools=TOOLS, cases=SUITE):
    rows, traces = [], {}
    for case in cases:
        FAULTS.clear(); FAULTS.update(case.get("faults", {}))
        tr = run_agent(case["task"], tools=tools, retry_transient=retry_transient)
        traces[case["id"]] = tr
        cost = (tr["prompt_tokens"] * PRICE["input"] + tr["completion_tokens"] * PRICE["output"]) / 1e6
        rows.append({"id": case["id"], **score(case, tr), "steps": tr["steps"], "tool calls": len(tr["calls"]),
                     "tokens": tr["prompt_tokens"] + tr["completion_tokens"], "cost $": cost,
                     "latency s": tr["llm_s"] + tr["tool_s"], "tool errors": tr["tool_errors"], "retries": tr["retries"],
                     "fault injected": bool(case.get("faults"))})
    FAULTS.clear()
    return pd.DataFrame(rows).set_index("id"), traces

v1, traces_v1 = run_suite(retry_transient=False)
v2, traces_v2 = run_suite(retry_transient=True)
print(v1.drop(columns=["cost $"]).round(3).to_string())
print(f"\nreal LLM calls so far: {CALLS['n']} (the v2 run re-used v1's cached responses wherever the conversation was identical)")

In [ ]:
for tid in ["T6", "T7", "T9"]:
    tr = traces_v1[tid]
    print(f"{tid} v1 calls={tr['calls']}\n    answer: {(tr['answer'] or '')[:120]}")
print(f"T6 v2 calls={traces_v2['T6']['calls']} retries={traces_v2['T6']['retries']}\n    answer: {traces_v2['T6']['answer'][:120]}")

In [ ]:
def summarise(df):
    s = {m: df[m].mean() for m in ["success", "tool selection", "arg accuracy", "exact trajectory", "unordered trajectory", "trajectory sim"]}
    s["recovery rate"] = df.loc[df["fault injected"], "success"].mean()
    s["$ per 1k successes"] = 1000 * df["cost $"].sum() / max(df["success"].sum(), 1)
    return s

summary = pd.DataFrame({"v1 (no retry)": summarise(v1), "v2 (retry once)": summarise(v2)})
print(summary.round(4).to_string())

metrics = ["success", "tool selection", "arg accuracy", "exact trajectory", "trajectory sim", "recovery rate"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(metrics))
for k, col in enumerate(summary.columns):
    axes[0].bar(x + (k - 0.5) * 0.38, summary.loc[metrics, col], width=0.38, label=col)
axes[0].set_xticks(x, metrics, rotation=25, ha="right"); axes[0].set_ylim(0, 1.05); axes[0].legend()
axes[0].set_title("Suite metrics: the retry moves success and recovery only")
heat = v2[["success", "tool selection", "arg accuracy", "exact trajectory"]].astype(float)
axes[1].imshow(heat.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
axes[1].set_xticks(range(heat.shape[1]), heat.columns, rotation=25, ha="right")
axes[1].set_yticks(range(len(heat)), [f"{i}: {c['task'][:28]}" for i, c in zip(heat.index, SUITE)], fontsize=8)
axes[1].set_title("v2 per task (green = pass)")
plt.tight_layout(); plt.show()

**Reading the results (offline stand-in):**
- **T6** is the recovery case. v1 passes the timeout back to the model, which can only report the error. v2's single retry turns it into a success. That is `ag19` in one row: classify errors as retryable or not, and retry only the retryable ones.
- **T7** gets the right tool with the **wrong argument**. The stand-in cannot see "new york" without capitals, so it defaults to another city. Tool-selection accuracy says 100% and argument accuracy catches it. That is why you measure both.
- **T9** fails on **tool selection**: "15% of 80" contains no arithmetic operator, so the keyword-matching stand-in never picks the calculator. A real model usually calls it with `0.15*80`, which is why the argument check compares values.
- With a real model, the per-task pattern will differ. That is the point of running the suite on every model, prompt or tool change.

## 6. Cost and latency per task, and cost per *successful* task

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(v2.index, v2["tokens"], color="#3b5bdb"); axes[0].set_ylabel("tokens"); axes[0].set_title("tokens per task (v2)")
lat = pd.DataFrame({tid: {"LLM": traces_v2[tid]["llm_s"], "tools": traces_v2[tid]["tool_s"]} for tid in v2.index}).T
axes[1].bar(lat.index, lat["LLM"], label="LLM time"); axes[1].bar(lat.index, lat["tools"], bottom=lat["LLM"], label="tool time")
axes[1].set_ylabel("seconds"); axes[1].set_title("latency per task, split by stage (v2)"); axes[1].legend()
plt.tight_layout(); plt.show()

"Cost per successful task" (`ev06`, `ag21`) is total cost divided by the number of successes, so a cheap agent that fails often is not as cheap as it looks. Take care with the classic example, though: *60% success at $0.10* versus *85% at $0.25*. Per success, that is $0.167 versus $0.294, so the cheap agent still wins **on API cost alone**. The expensive agent wins only once failures have a price, for example a human handling each escalation. Always put the cost of failure into the comparison.

In [ ]:
def cost_per_task(success_rate, api_cost, escalation_cost):
    return {"API $ per success": api_cost / success_rate,
            "all-in $ per task": api_cost + (1 - success_rate) * escalation_cost}

for esc in [0.0, 1.0, 5.0]:
    a, b = cost_per_task(0.60, 0.10, esc), cost_per_task(0.85, 0.25, esc)
    print(f"escalation ${esc:.0f}: cheap agent {a['all-in $ per task']:.3f} vs better agent {b['all-in $ per task']:.3f} per task "
          f"(API per success {a['API $ per success']:.3f} vs {b['API $ per success']:.3f})")
assert cost_per_task(0.60, 0.10, 0)["API $ per success"] < cost_per_task(0.85, 0.25, 0)["API $ per success"]
assert cost_per_task(0.60, 0.10, 5)["all-in $ per task"] > cost_per_task(0.85, 0.25, 5)["all-in $ per task"]

## 7. An LLM judge for final answers

Substring checks are cheap and exact, but brittle: "12.0" is not "12", and "arriving today" is not "out for delivery". A reference-guided judge (see [notebook 02](02_llm_as_judge_and_its_biases.ipynb)) handles paraphrase. Run both. The **disagreements** are what a human should look at.

In [ ]:
class AnswerGrade(BaseModel):
    reasoning: str = Field(description="One sentence.")
    correct: bool = Field(description="True if the final answer states the expected facts for the task.")

def judge_answer(case, answer):
    user = f"TASK: {case['task']}\nEXPECTED FACTS: {', '.join(case['facts']) or '(no tool needed; any friendly reply)'}\nFINAL ANSWER: {answer}"
    resp = client.chat.completions.parse(model=MODEL, temperature=0, response_format=AnswerGrade, messages=[
        {"role": "system", "content": "Grade an assistant's final answer against the expected facts. Wording does not matter."},
        {"role": "user", "content": user}])
    return resp.choices[0].message.parsed

judged = pd.DataFrame([{"id": c["id"], "substring check": bool(v2.loc[c["id"], "success"]),
                        "LLM judge": judge_answer(c, traces_v2[c["id"]]["answer"]).correct} for c in SUITE]).set_index("id")
judged["agree"] = judged["substring check"] == judged["LLM judge"]
print(judged.to_string())
print(f"agreement: {judged.agree.mean():.0%}; disagreements to review by hand: {judged.index[~judged.agree].tolist()}")
if PROVIDER == "offline":
    print("(offline stand-in: the judge's verdict is canned, not reasoned. With a real judge, disagreements are usually "
          "paraphrases the substring check missed, or confident wrong answers it caught.)")

## 8. Golden-trajectory regression tests

Record the trajectories of a known-good version. Then, on every change to the prompt, the tools or the model, assert that the new trajectories still match. The check is on **structure** (tool names and normalised arguments), never on the exact wording of the answer (`ev22`).

The classic regression is invisible in a code review: someone "tidies" a tool's name and description. `ag06`'s whole point is that tool descriptions, not prompts, often explain a 2× gap between agents. Below, the order tool is renamed to `svc_fetch_v2` with the description "Internal endpoint." and the test runs again.

In [ ]:
GOLDEN_TRAJ = {c["id"]: [n for n, _ in c["expect"]] for c in SUITE}      # reviewed expected tool sequences

def regression_test(traces, golden=GOLDEN_TRAJ):
    """Tasks whose tool-name sequence (as a multiset, so parallel order is free) differs from the golden one."""
    return {tid: (golden[tid], [n for n, _ in tr["calls"]]) for tid, tr in traces.items()
            if sorted(golden[tid]) != sorted(n for n, _ in tr["calls"])}

# the checker itself, on hand-made traces
assert regression_test({"T3": {"calls": [("lookup_order", {})]}}) == {}
assert regression_test({"T3": {"calls": []}}) == {"T3": (["lookup_order"], [])}

print("current version, known failures:", regression_test(traces_v2))

BAD_TOOLS = [TOOLS[0], TOOLS[1], tool_schema("svc_fetch_v2", "Internal endpoint.", "key", "record key")]
REGISTRY["svc_fetch_v2"] = lambda key: lookup_order(key)
order_cases = [c for c in SUITE if c["id"] in ("T3", "T6")]
_, traces_bad = run_suite(retry_transient=True, tools=BAD_TOOLS, cases=order_cases)
golden_bad = {c["id"]: ["svc_fetch_v2"] for c in order_cases}          # same behaviour, new tool name
print("after the 'harmless' rename:", regression_test(traces_bad, golden_bad) or "no regressions")

Offline, the stand-in stops calling the tool at all once its name and description no longer match the request, and the test catches it before deploy. A real model may still guess from context, or may not. Either way, the suite tells you instead of your users. Gate on a **pass rate** over several runs rather than all-green, because agents are not perfectly deterministic even at temperature 0 (`ag28`).

## 9. Reliability: pass@k versus pass^k

Run each task n times. With c successes:
- **pass@k** = 1 − C(n−c, k)/C(n, k): *at least one* of k attempts succeeds. It is optimistic, and it suits a user who can retry.
- **pass^k** = C(c, k)/C(n, k): *all* k attempts succeed. τ-bench introduced it to measure consistency. It is the number that matters for an agent that must work every time it runs.

Below, the weather and order tools fail at random 30% of the time (seeded). Three tasks run 6 times each, with and without the retry.

In [ ]:
def pass_at_k(n, c, k):
    return 1.0 if n - c < k else 1 - comb(n - c, k) / comb(n, k)

def pass_hat_k(n, c, k):
    return comb(c, k) / comb(n, k)

assert pass_hat_k(4, 4, 3) == 1.0 and pass_hat_k(4, 3, 4) == 0.0 and abs(pass_hat_k(4, 2, 2) - 1 / 6) < 1e-12

N_TRIALS, flaky_cases = 6, [c for c in SUITE if c["id"] in ("T1", "T3", "T4")]
successes = {}
for retry in (False, True):
    FLAKY.update(p=0.3, rng=np.random.default_rng(42))
    successes[retry] = [sum(bool(run_suite(retry, cases=[c])[0].loc[c["id"], "success"]) for _ in range(N_TRIALS)) for c in flaky_cases]
FLAKY["p"] = 0.0
print("successes out of", N_TRIALS, "per task (T1, T3, T4):", {("retry" if r else "no retry"): s for r, s in successes.items()})

ks = range(1, N_TRIALS + 1)
fig, ax = plt.subplots(figsize=(8, 3.8))
for retry, style in [(False, "--"), (True, "-")]:
    label = "retry once" if retry else "no retry"
    ax.plot(ks, [np.mean([pass_at_k(N_TRIALS, c, k) for c in successes[retry]]) for k in ks], "o" + style, label=f"pass@k, {label}")
    ax.plot(ks, [np.mean([pass_hat_k(N_TRIALS, c, k) for c in successes[retry]]) for k in ks], "s" + style, label=f"pass^k, {label}")
ax.set_xlabel("k"); ax.set_ylabel("probability"); ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
ax.set_title("pass@k rises with k, pass^k falls: flakiness hides in pass@k"); plt.show()
print(f"real LLM calls in this notebook: {CALLS['n']}")

## 10. Public agent benchmarks (brief, and hedged)

| Benchmark | What the agent does | How success is checked | Watch out for |
|---|---|---|---|
| **τ-bench** (Sierra, 2024) | customer-service tasks in retail and airline domains, talking to an LLM-simulated user, with tools and policies | the final **database state** matches the goal; introduced **pass^k** | simulated users are more cooperative than real ones; a later τ²-bench extends the domains |
| **SWE-bench** (Princeton, 2023) | resolve real GitHub issues in Python repositories by editing code | hidden unit tests that the fix must make pass (and must not break others) | *Verified* (500 human-checked tasks) and *Lite* subsets exist; the harness and scaffolding matter as much as the model |
| **GAIA** (2023) | general-assistant questions needing browsing, files and multi-step reasoning, at 3 difficulty levels | short answers, exact match | at release, humans scored far above tool-using GPT-4 |
| **WebArena** (CMU, 2023) | tasks on self-hosted realistic websites (shop, forum, GitLab, CMS, maps) | functional checks of the resulting site state | long-horizon and brittle; the environment setup is heavy |

What they tell you: roughly how capable a model plus scaffold is at *that kind* of task. What they do not tell you: how your agent does on your tools, your data and your users, or whether scores are inflated by contamination (`ev20`). Build a private suite like this one. It is the number you ship on.

## Try it yourself

**1. Add a case.** Add `T10`, "How much is 250 divided by 5?", expecting `calculator` with a value of 50. Run just that case through `run_suite` and `score`. Why does it pass or fail offline?

In [ ]:
# Solution — try it yourself first, then run this
t10 = {"id": "T10", "task": "How much is 250 divided by 5?", "expect": [("calculator", {"expression": "250/5"})], "facts": ["50"]}
df10, tr10 = run_suite(retry_transient=True, cases=[t10])
print(df10[["success", "tool selection", "arg accuracy"]].to_string(), "\ncalls:", tr10["T10"]["calls"])
# Offline, the word 'divided' has no operator symbol, so the keyword stand-in skips the calculator
# (the same failure class as T9). A real model calls calculator('250/5'). Keep this case in the suite.
# Gotcha: the canned reply echoes the question, and '250' contains '50'. A plain `'50' in answer` check
# would have scored this failure a SUCCESS. That is why has_fact() matches whole tokens only.
print("naive substring check would say:", "50" in (tr10["T10"]["answer"] or ""))

**2. Subset and superset matching.** Some teams accept extra harmless calls (superset) or require only certain key calls (subset). Write `trajectory_match(expected, actual, mode)` for the modes "strict", "unordered", "subset" (every expected call appears) and "superset" (every actual call was expected). Test it on hand-made examples.

In [ ]:
# Solution — try it yourself first, then run this
from collections import Counter

def trajectory_match(expected, actual, mode="strict"):
    e, a = Counter(expected), Counter(actual)
    return {"strict": list(expected) == list(actual), "unordered": e == a,
            "subset": not (e - a), "superset": not (a - e)}[mode]

exp_, act_ = ["get_weather", "calculator"], ["calculator", "get_weather", "get_weather"]
print({m: trajectory_match(exp_, act_, m) for m in ["strict", "unordered", "subset", "superset"]})
assert trajectory_match(exp_, act_, "subset") and not trajectory_match(exp_, act_, "superset")

**3. The step cap and a repeat-call guard.** `tl22` asks for four guards: a step cap, a cost ceiling, repeat-call detection and a named degraded outcome. `run_agent` has the step cap and `llm()` has the budget. Write `repeated_calls(trace)` to flag any (tool, args) pair called more than once, and test it on a fake trace.

In [ ]:
# Solution — try it yourself first, then run this
def repeated_calls(trace):
    seen = Counter((n, json.dumps(a, sort_keys=True)) for n, a in trace["calls"])
    return [call for call, cnt in seen.items() if cnt > 1]

fake = {"calls": [("lookup_order", {"order_id": "ORD-1"}), ("lookup_order", {"order_id": "ORD-1"}), ("calculator", {"expression": "1+1"})]}
print(repeated_calls(fake))
assert repeated_calls(fake) == [("lookup_order", '{"order_id": "ORD-1"}')] and repeated_calls(traces_v2["T4"]) == []

## Say it in an interview

**30-second answer (`ag21`):** "A model eval grades one answer. An agent eval grades a trajectory. I run a fixed task suite against mocked, deterministic tools and score task success against the end state or the required facts, tool-selection accuracy, argument correctness, and trajectory match (exact, unordered, or an edit-distance similarity). I also score steps, tokens, latency and cost per *successful* task, and recovery when I inject tool failures. In CI I gate on a pass rate over repeated runs, keep golden trajectories as regression tests, and track pass^k when the agent must be reliable, not just lucky."

**Numbers and definitions:**
- Trajectory similarity = 1 − Levenshtein(expected, actual) / max length.
- pass@k = 1 − C(n−c,k)/C(n,k), the chance that *any* of k attempts succeeds. pass^k = C(c,k)/C(n,k), the chance that *all* do.
- Cost per success = total cost / successes. The all-in cost adds (1 − success) × the cost of a failure.

**Follow-ups:**
- *Why mock tools?* Otherwise you measure the network and the data, not the agent. Record fixtures and replay them.
- *Two agents with the same model score 71% vs 38%?* (`ag06`) Look at tool descriptions, parameter schemas (enums), error messages, observation size, history policy and stop conditions, not the prompt.
- *How do you handle a failing tool?* (`ag19`) Separate retryable from terminal errors, cap retries, humanise error messages, and measure the recovery rate.
- *Is SWE-bench a good proxy for our coding agent?* Only partly: it measures issue resolution in popular Python repos, with a specific harness. Build a private suite.

**Traps:**
- Grading the agent's *claim* ("I issued the refund") instead of the state.
- Exact-string answer checks.
- An all-green gate that flakes and gets disabled.
- Quoting pass@k when the agent must succeed every time.
- Forgetting that failures have a cost.

## Next

- [05 · A/B testing and statistics for evals](05_ab_testing_and_statistics_for_evals.ipynb): is v2 really better on 9 tasks?
- [02 · LLM-as-judge and its biases](02_llm_as_judge_and_its_biases.ipynb)
- [Agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) · [Tool design and function schemas](../15_agentic_ai/02_tool_design_and_function_schemas.ipynb) · [Guardrails, HITL and failure modes](../15_agentic_ai/06_guardrails_hitl_and_failure_modes.ipynb)
- [Retries, backoff, rate limits, circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb)
- [Eval harness from scratch](../17_interview_coding_rounds/05_eval_harness_from_scratch.ipynb) · [Build an agent in 30 minutes](../17_interview_coding_rounds/02_build_an_agent_in_30_minutes.ipynb)
- Theory: [agentic_ai_flow course](../../agentic_ai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)